# Single-event waveform inspection

Load a shotgun waveform HDF5, pick a case, and show:

1. Detector-summed waveform (linear + log y — tail is invisible in linear).
2. **3D scatter of all sensors coloured by integrated charge** (linear + log).
3. Same 3D scatter coloured by per-sensor first-hit time — shows the
   Cherenkov / time-of-flight geometry.
4. Top-10 brightest sensor waveforms with a dashed marker at direct-light ToF.


In [ ]:
import sys
sys.path.append('../../../../')  # notebooks/ → photon_shotgun/ → production/ → lucid/ → repo root

import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from lucid.production.photon_shotgun.io import (
    load_shotgun_waveform, load_shotgun_per_photon,
)
from lucid.production.photon_shotgun.viz import (
    scatter_sensors_3d, plot_hist_lin_log,
)
from lucid.geometry import generate_detector

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 5)


def _resolve_geom(meta):
    geom_path = meta.get('detector_config', '')
    if isinstance(geom_path, bytes):
        geom_path = geom_path.decode()
    for candidate in ['../../../../' + geom_path, geom_path]:
        if os.path.exists(candidate):
            return candidate
    return geom_path


In [ ]:
WAVE_PATH = '../../../../runs/shotgun_SK_1k_waveform.h5'
CASE = 0

out = load_shotgun_waveform(WAVE_PATH)
meta = dict(out['meta'])
num_sensors = int(meta['num_sensors'])
n_time_bins = int(meta['n_time_bins'])
bin_w = float(meta['bin_width_ns'])

print(f"file contains {int(meta['n_cases'])} cases × {int(meta['n_photons'])} photons")
print(f"waveform shape per case: ({num_sensors}, {n_time_bins})   "
      f"window={meta['window_ns']} ns, bin={bin_w} ns")
print(f"total detected: {int(out['n_detected'].sum()):,}   "
      f"dropped: {int(out['n_dropped'].sum())}")


## Reconstruct one case from COO

COO storage lets us load just the nonzero entries for a single case without
materializing the full ``(n_cases, num_sensors, n_time_bins)`` tensor.


In [ ]:
m = out['case_idx'] == CASE
sid, tb, ch = out['sensor_id'][m], out['time_bin'][m], out['charge'][m]
wf_case = np.zeros((num_sensors, n_time_bins), dtype=np.float32)
wf_case[sid, tb] = ch

charge_per_sensor = wf_case.sum(axis=1)
has_hit = wf_case > 0
first_bin = np.where(has_hit.any(axis=1), np.argmax(has_hit, axis=1), -1)
first_time = np.where(first_bin >= 0, first_bin * bin_w, np.nan)

print(f"case {CASE}: detected={int(out['n_detected'][CASE])}  "
      f"sensors hit={(charge_per_sensor > 0).sum()}  "
      f"total charge={charge_per_sensor.sum():.1f}")


## Detector-summed waveform


In [ ]:
total_time = wf_case.sum(axis=0)
t_axis = np.arange(n_time_bins) * bin_w

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
for ax, yscale in zip(axes, ('linear', 'log')):
    ax.step(t_axis, total_time, where='mid', color='steelblue')
    ax.set_yscale(yscale)
    ax.set_xlabel('time (ns)')
    ax.set_ylabel(f'total charge ({yscale})')
    ax.grid(alpha=0.3)
fig.suptitle(f'case {CASE} — detector-summed waveform')
fig.tight_layout()
plt.show()


## Sensor geometry


In [ ]:
det = generate_detector(_resolve_geom(meta))
sensor_points = np.asarray(det.all_points)
print(f"{sensor_points.shape[0]} sensors; z range [{sensor_points[:, 2].min():.1f}, {sensor_points[:, 2].max():.1f}]")

src = out.get('source')
source_origin = np.asarray(src.origins)[CASE, 0] if src is not None else None
if source_origin is not None:
    print(f"source origin for case {CASE}: {source_origin}")


## Integrated charge per sensor (3D) — linear + log


In [ ]:
fig = plt.figure(figsize=(14, 6))
ax1 = fig.add_subplot(121, projection='3d')
scatter_sensors_3d(sensor_points, charge_per_sensor, ax=ax1, log=False,
                   title=f'case {CASE} — integrated charge (linear)',
                   cbar_label='charge (PE)',
                   source_origin=source_origin)
ax2 = fig.add_subplot(122, projection='3d')
scatter_sensors_3d(sensor_points, charge_per_sensor, ax=ax2, log=True,
                   title=f'case {CASE} — integrated charge (log)',
                   cbar_label='charge (PE)',
                   source_origin=source_origin)
plt.show()


## First-hit time per sensor (3D)


In [ ]:
first_time_plot = np.nan_to_num(first_time, nan=0.0)
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')
scatter_sensors_3d(sensor_points, first_time_plot, ax=ax, log=False,
                   cmap='viridis', title=f'case {CASE} — first-hit time',
                   cbar_label='time (ns)',
                   source_origin=source_origin)
plt.show()


## Top-10 brightest sensor waveforms

Dashed red marker: expected direct-light time-of-flight from the source
(``|r_sensor − r_source| / c_medium``).


In [ ]:
c_med = 0.2253  # m/ns in water
top_idx = np.argsort(charge_per_sensor)[-10:][::-1]
fig, axes = plt.subplots(5, 2, figsize=(11, 10), sharex=True)
for ax, sidx in zip(axes.flat, top_idx):
    ax.step(t_axis, wf_case[sidx], where='mid')
    if source_origin is not None:
        d = float(np.linalg.norm(sensor_points[sidx] - source_origin))
        ax.axvline(d / c_med, color='red', lw=0.8, alpha=0.4, ls='--')
    ax.set_title(f'sensor {sidx}  (Q={charge_per_sensor[sidx]:.1f})')
    ax.set_ylabel('charge')
    ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel('time (ns)')
fig.suptitle(f'case {CASE} — top-10 brightest sensors (red dashed = direct-light ToF)')
fig.tight_layout()
plt.show()
